In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from numpy.linalg import eig
import pandas as pd

#import matplotlib  
#matplotlib.use('Agg')  # Use a non-GUI backend

In [2]:
cat='C6'
path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/zbox/'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'

#df = pd.DataFrame(columns=['nrep', '#hops' 'Dnonhop_mean', 'Dnonhop_se', 'D_hop', 'Dtot'])

In [3]:
def oh_xyz(nrep, nsteps):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)

    #extracting x,y, z cordinates from oh_id.dat files
    x_oh = np.zeros((nrep,nsteps))
    y_oh = np.zeros((nrep,nsteps))
    z_oh = np.zeros((nrep,nsteps))
    indx_oh= np.zeros((nrep,nsteps))

    for i in range(nrep):
        with open(path+ f'oh_id_z17.5.dat', 'r') as oh_id:
            xyz= oh_id.readlines()[:nsteps]
            
            for j in range(len(xyz)):   
                indx_oh[i,j]=int(xyz[j].split()[1])
                x_oh[i,j]=float(xyz[j].split()[2])
                y_oh[i,j]=float(xyz[j].split()[3])
                z_oh[i,j]=float(xyz[j].split()[4])
                
    #dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)
    return(x_oh, y_oh, z_oh, indx_oh)




In [4]:
nrep=1

nsteps=2000

x_oh, y_oh, z_oh, indx_oh= oh_xyz(nrep, nsteps)

In [5]:
dtime = np.zeros(nsteps)
D1x=np.zeros(nrep)
D1y=np.zeros(nrep)
D1=np.zeros(nrep)
rx=[]
ry=[]
t=[]
ax=[]
ay=[]
t_hop=[]
tau=[]
for ll in range(nrep):
    int_indx= indx_oh[ll,0]
    int_x= x_oh[ll,0]
    int_y= y_oh[ll,0]
    int_z= z_oh[ll,0]
    int_t=0
    hop=0
    for jj in range(nsteps):
        dtime[jj]= np.round(jj*0.01, 2)
        
        if indx_oh[ll,jj]==int_indx:
            pass
            #print('nonhop',ll, jj, indx_oh[ll,jj],dtime[jj], x_oh[ll,jj])
        else:
            hop=hop+1
            #t_hop.append(dtime[jj])
            rx.append((x_oh[ll,jj-1]-int_x)**2)
            ry.append((y_oh[ll,jj-1]-int_y)**2)
            t.append(dtime[jj-1]-int_t)
            #print('hop',ll, jj, indx_oh[ll,jj])
            print(int_t, int_indx, int_x, int_y)
            print(dtime[jj-1], indx_oh[ll,jj-1], x_oh[ll,jj-1], y_oh[ll,jj-1])     
            print(t[-1], rx[-1], ry[-1])   


            if hop>1:
                ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                #print(ll,jj)
                #print(x_oh[ll,jj],x_oh[ll,jj-1])
                #print(dtime[jj-1],int_t)
                t_hop.append(dtime[jj-1]-int_t)

            
            int_indx= indx_oh[ll,jj]
            int_x= x_oh[ll,jj]
            int_y= y_oh[ll,jj]
            int_z= z_oh[ll,jj]
            int_t=dtime[jj]
            #print(hop,int_t)

            #print(indx_oh[ll,jj], x_oh[ll,jj], x_oh[ll,jj-1], dtime[jj],int_t)
            
    print(ll,hop)
    rx.append((x_oh[ll,-1]-int_x)**2)
    ry.append((y_oh[ll,-1]-int_y)**2)
    t.append(dtime[-1]-int_t)
    
    D1x[ll]=(np.mean(rx))/(2*np.mean(t))
    D1y[ll]=(np.mean(ry))/(2*np.mean(t))
    D1[ll]=(np.mean(rx)+np.mean(ry))/(4*np.mean(t))

dmean= np.mean(D1)
dstd= np.std(D1)
std_err= dstd/np.sqrt(nrep)

D2x=(np.mean(ax))/(2*np.mean(t_hop))
D2y=(np.mean(ay))/(2*np.mean(t_hop))
D2=(np.mean(ax)+np.mean(ay))/(4*np.mean(t_hop))
D=dmean+D2
                    
print('nohops:',dmean, std_err)
print('hops:',D2)
print('Total:',D)
print(D1)

0 352.0 0.83161935 -3.98553384
0.58 352.0 -1.43395012 -5.09689348
0.58 5.132805023396081 1.2351202494209303
0 1
nohops: 3.1454756224766247 0.0
hops: nan
Total: nan
[3.14547562]


C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [6]:
len(t_hop)+nrep

1

In [7]:
dmean= np.mean(D1)
dstd= np.std(D1)
std_err= dstd/np.sqrt(nrep)
dmean, dstd, std_err

(3.1454756224766247, 0.0, 0.0)

In [8]:
c2=0.9627078511609803
c4= 0.8756260083700356
c6=0.738363956584734